# Importe de librerías

In [ ]:
import os
import email
from email import policy
from bs4 import BeautifulSoup
from email.message import EmailMessage
import re
import pandas as pd
from datetime import datetime
from email.utils import parsedate_to_datetime
import sys
import email
from email import policy
from email.parser import BytesParser
from email.utils import parsedate_to_datetime
import re
from html import unescape
from zoneinfo import ZoneInfo
import warnings
warnings.simplefilter(action='ignore', category=Warning)

# Lectura de correos

## Función que lee la fecha de la tabla de cada respuesta en los correos leyendo el HTML

In [ ]:
def extraer_fecha_envio(tag):
        #Lee el siguiente texto después de un tag de HTML como "Enviado él: " o "Sent: "
        siguiente_texto = tag.next_sibling
        while siguiente_texto:
            if isinstance(siguiente_texto, str):
                texto = unescape(siguiente_texto.strip().lower()) #Extrae texto
                #Extracción de las fechas con expresiones regulares
                match_es = re.search(r'(\d{1,2})\s+de\s+(\w+)\s+de\s+(\d{4})', texto)
                if match_es:
                    dia, mes_texto, año = match_es.groups()
                    meses_es = {
                        'enero': '01', 'febrero': '02', 'marzo': '03', 'abril': '04',
                        'mayo': '05', 'junio': '06', 'julio': '07', 'agosto': '08',
                        'septiembre': '09', 'octubre': '10', 'noviembre': '11', 'diciembre': '12'
                    }
                #Consolidación de fechas
                    mes = meses_es.get(mes_texto, '01')
                    return f"{año}-{mes}-{int(dia):02d}"

                # Extracción de fecha en inglés
                match_en = re.search(r'(\w+)\s+(\d{1,2}),\s+(\d{4})', texto)
                if match_en:
                    mes_texto, dia, año = match_en.groups()
                    meses_en = {
                        'january': '01', 'february': '02', 'march': '03', 'april': '04',
                        'may': '05', 'june': '06', 'july': '07', 'august': '08',
                        'september': '09', 'october': '10', 'november': '11', 'december': '12'
                    }
                    mes = meses_en.get(mes_texto, '01')
                    return f"{año}-{mes}-{int(dia):02d}"
            siguiente_texto = siguiente_texto.next_sibling
        return None

## Función que lee la fecha del último correo enviado

In [ ]:
    #Lectura de la fecha del último correo enviado, ya que el HTML de este no incluye elementos que se puedan taggear para extraer fechas como los anteriores
def parse_eml_date(path):
        with open(path, 'rb') as f:
            msg = BytesParser(policy=policy.default).parse(f)
        date_hdr = msg['Date']
        if date_hdr:
            try:
                dt = parsedate_to_datetime(date_hdr)
                return ('Date header', dt)
            except Exception:
                pass
        received = msg.get_all('Received', []) #Extracción de la fecha según el received del mensaje
        if received:
            last = received[-1]
            m = re.search(r';\s*(.+)$', last)
            if m:
                date_str = m.group(1).strip()
                try:
                    dt = parsedate_to_datetime(date_str)
                    return ('Last Received header', dt)
                except Exception:
                    pass
            parts = re.split(r'\(|\)|;', last)
            for part in reversed(parts):
                part = part.strip()
                try:
                    dt = parsedate_to_datetime(part)
                    if dt:
                        return ('Received (heuristic)', dt)
                except Exception:
                    pass
        return (None, None)

## Función que extrae información de correos y consolida los dataframes

In [ ]:
def LecturaCorreos(correo):

    tipo_fecha, fecha = parse_eml_date(correo)
    if fecha:
        fecha_bogota = fecha.astimezone(ZoneInfo("America/Bogota"))
        solo_fecha = fecha_bogota.date()  # Extracción de fecha, eliminación de la hora
        # Abrir el archivo de correo
    with open(correo, 'rb') as f:
        msg = email.message_from_binary_file(f, policy=policy.default)
    # Lista para acumular contenido HTML
    html_parts = []
    timestamp = os.path.getmtime(correo)
    # Recorrer todas las partes del mensaje
    for part in msg.walk():
        content_type = part.get_content_type()
        filename = part.get_filename()
        # Solo partes HTML que no sean archivos adjuntos
        if content_type == "text/html" and filename is None:
            try:
                html_parts.append(part.get_content())
            except:
                pass  # Evita errores si alguna parte no se puede leer

    # Unir todas las partes HTML
    html_content = "\n".join(html_parts)
    # Extraer todas las tablas del HTML
    tables = []
    fechas_por_tabla = []
    if html_content:
        soup = BeautifulSoup(html_content, "html.parser") #Recorre HTML
        for tag in soup.find_all("b"):
            texto = tag.get_text(strip=True).lower()
        fecha_actual = None

        for tag in soup.find_all(["b", "table"]): #Reconoce las etiquetas de tabla
            texto = tag.get_text(strip=True).lower()
            # Detectar bloque "Sent:" y extraer fecha
            if "sent:" in texto or "enviado el:" in texto:
                fecha_actual = extraer_fecha_envio(tag)
            # Si se encuentra tabla, asigna la fecha
            if tag.name == "table":
                try:
                    df = pd.read_html(str(tag))[0]
                    #Filtra tablas relevantes para el registro de horas
                    columnas_texto = " ".join(df.astype(str).values.flatten()).lower()
                    if not any(palabra in columnas_texto for palabra in [
                        "horas trabajadas", "tiempo ejecutado", "estimación de tiempos",
                        "asignaciones adicionales", "estado", "tiempo adicional"
                    ]):
                        continue
                    fechas_por_tabla.append(fecha_actual)
                    tables.append(df)
                except:
                    pass

#Rellena la primera fecha de la tablas de fechas extraidas con HTML por la extraída en el último correo enviado
    if fechas_por_tabla and fechas_por_tabla[0] is None: 
        fecha_str = solo_fecha.strftime("%Y-%m-%d")
        fechas_por_tabla[0] = fecha_str

#Extrae el sender del correo
    sender = msg['From'] 
    
    processed_tables = []
    for df in tables:
        try:
            # Verifica si la tabla contiene 'Id Backlog' en la primera columna
            if 'Id Backlog' in df.iloc[:, 0].values:
                df['sender'] = sender

                # Recorta desde 'Id Backlog' hasta 'ASIGNACIONES ADICIONALES'
                idx_start = df[df.iloc[:, 0] == 'Id Backlog'].index[0]
                df = df.loc[idx_start:]

                idx_end = df[df.iloc[:, 0] == 'ASIGNACIONES ADICIONALES'].index[0]
                df = df.loc[:idx_end - 1]

                # Elimina filas con 'Nombre'
                df = df[df.iloc[:, 0] != 'Nombre']

                # Extrae número de la segunda columna
                df.iloc[:, 1] = df.iloc[:, 1].astype(str).str.extract(r'(\d+\.\d+|\d+)', expand=False).astype(float)

                # Extrae valor de backlog
                backlog = df.loc[df.iloc[:, 0] == 'Id Backlog', df.columns[1]].values[0]
                df['Backlog'] = backlog

                # Elimina fila de 'Id Backlog'
                df = df[df.iloc[:, 0] != 'Id Backlog']

                # Reemplaza comas por puntos en todo el DataFrame
                df = df.applymap(lambda x: x.replace(',', '.') if isinstance(x, str) else x)

                # Renombra columnas
                df.columns = [
                    'Elemento',
                    'Horas Estimadas de Desarrollo',
                    'Tiempo adicional estimado',
                    'Horas Trabajadas',
                    'Estado',
                    'sender',
                    'Backlog'
                ]

                processed_tables.append(df)
        except Exception as e:
            continue  # Si falla, pasa a la siguiente tabla

    consolidated = []

    for i, df in enumerate(processed_tables):
        try:
            # Renombrar columnas si no están ya renombradas
            df.columns = [
                'Elemento',
                'Horas Estimadas de Desarrollo',
                'Tiempo adicional estimado',
                'Horas Trabajadas',
                'Estado',
                'sender',
                'Backlog'
            ]

            #Join de fechas por tabla de respuesta
            fecha_correo = fechas_por_tabla[i]
            df['Fecha del correo'] = fecha_correo

            # Extracción de horas trabajadas. Se hace necesario uso de expresiones regulares por caraceteres como "Hoy" o paréntesis
            def extract_hours(text):
                if pd.isna(text):
                    return None
                text = str(text).lower()
                matches = re.findall(r'(\d+(?:\.\d+)?)\s*horas', text)
                total = sum(float(m) for m in matches)
                return total if total > 0 else None

            df['Horas Ejecutadas de Desarrollo'] = df['Horas Trabajadas'].apply(extract_hours)
            df['Tiempo adicional estimado'] = df['Tiempo adicional estimado'].apply(extract_hours)

            # Extracción de horas trabajadas hoy
            def extract_horas_hoy(text):
                if pd.isna(text):
                    return 0.0
                text = str(text).lower()
                match = re.search(r'(\d+(?:\.\d+)?)\s*horas\s*hoy', text)
                return float(match.group(1)) if match else 0.0

            df['Horas trabajadas hoy'] = df['Horas Trabajadas'].apply(extract_horas_hoy)

            # Selección de columnas finales
            df_clean = df[[
                'Elemento',
                'Backlog',
                'sender',
                'Fecha del correo',
                'Horas Estimadas de Desarrollo',
                'Tiempo adicional estimado',
                'Horas Ejecutadas de Desarrollo',
                'Horas trabajadas hoy',
                'Estado'
            ]]
            consolidated.append(df_clean)
        except Exception as e:
            continue
    # Unión de dataframes extraídos
    df_total = pd.concat(consolidated, ignore_index=True)
    if consolidated:
        df_total = pd.concat(consolidated, ignore_index=True)
        return df_total
    else:
        return pd.DataFrame()  # Devuelve un DataFrame vacío si no se extrajo nada


## Lectura de correos


In [ ]:
# Ruta de la carpeta que quieres recorrer
carpeta = "./proyectopiojoimg"
patron_exclusion = re.compile(r"_\d+\.[^.]+$")
# Lista para guardar todos los DataFrames extraídos
dataframes = []

# Recorremos todos los archivos en la carpeta
for archivo in os.listdir(carpeta):
    ruta_completa = carpeta+'/'+archivo
    # Verificamos que sea un archivo .eml
    if os.path.isfile(ruta_completa) and archivo.endswith(".eml"):
        if not patron_exclusion.search(archivo):
            try:
                # Aplicamos tu función personalizada
                df = LecturaCorreo(ruta_completa)
                dataframes.append(df)
                print(archivo,"procesado")
            except Exception as e:
                print(f"❌ Error procesando {archivo}: {e}")

# Concatenamos todos los DataFrames en uno solo
if dataframes:
    df_consolidado = pd.concat(dataframes, ignore_index=True)
    print("✅ DataFrame final creado con", len(df_consolidado), "filas.")
else:
    print("⚠️ No se extrajo ninguna tabla.")